# LLMs for Business Valuation

## Valuation Fundamentals: DCF Foundation & Challenges


**The Core DCF Formula: Discount Cash Flows**

$$\text{Enterprise Value} = \mathbb{E} \Big(\sum_{t=1}^{T}\frac{\text{FCFF}_t}{(1+WACC)^t} + \frac{\text{TV}_T}{(1+WACC)^T}\Big)$$

We will focus mainly on the Free Cash Flow to the Firm (cash that is available to all stakeholders) and the Free Cash Flow to Equity (cash available to equity holders after all expenses, debt payments, and reinvestment).

$$FCFF_t = EBIT_t(1-\tau) + D\&A_t - CapEx_t - \Delta NWC_t$$

Where:

- $FCFF_t$ = Free Cash Flow to the Firm in period $t$
- $WACC$ = Weighted Average Cost of Capital  
- $TV_T$ = Terminal Value at forecast end
- $\tau$ = tax rate, $D\&A$ = depreciation & amortization

**DCF Challenges:**
- Requires detailed forecasts for 5-10 years
- Terminal value often represents 60-80% of total value  
- Highly sensitive to growth and discount rate assumptions
- Time-intensive and subjective

**Why Multiples Matter:**
- Quick market-based alternative
- Reflects current investor sentiment  
- Useful for cross-sectional comparison
- Can validate DCF assumptions

## The Mathematical Connection: DCF to Multiples (Part 1)

**Step 1: Start with the perpetuity DCF formula**

For a company in steady state with constant growth $g$:

$$EV = \frac{FCFF_1}{WACC - g}$$

**Step 2: Express FCFF in terms of EBITDA**

Starting from our FCFF formula:
$$FCFF = EBIT(1-\tau) + D\&A - CapEx - \Delta NWC$$

Since $EBIT = EBITDA - D\&A$, we can substitute:
$$FCFF = (EBITDA - D\&A)(1-\tau) + D\&A - CapEx - \Delta NWC$$

Expanding and simplifying:
$$FCFF = EBITDA(1-\tau) - D\&A(1-\tau) + D\&A - CapEx - \Delta NWC$$
$$FCFF = EBITDA(1-\tau) + D\&A \cdot \tau - CapEx - \Delta NWC$$

## The Mathematical Connection: DCF to Multiples (Part 2)

**Step 3: Express each component as a ratio of EBITDA**

Let's define these key ratios:
- $\kappa = \frac{CapEx}{EBITDA}$ (CapEx intensity)
- $\delta = \frac{D\&A}{EBITDA}$ (D&A rate)
- $\omega = \frac{\Delta NWC}{EBITDA}$ (Working capital intensity)

**Step 4: Substitute back into the perpetuity formula**

$$FCFF = EBITDA[(1-\tau) + \delta \cdot \tau - \kappa - \omega]$$

Therefore:
$$EV = \frac{EBITDA[(1-\tau) + \delta \cdot \tau - \kappa - \omega]}{WACC - g}$$

**Step 5: Solve for the EV/EBITDA multiple**

$$\frac{EV}{EBITDA} = \frac{(1-\tau) + \delta \cdot \tau - \kappa - \omega}{WACC - g}$$

**Key Insight:** The EV/EBITDA multiple depends on tax rate, CapEx intensity, growth, and risk!

## Practical Example & Sensitivity Analysis

**Given company assumptions:**
- Tax rate: $\tau = 25\%$
- CapEx intensity: $\kappa = 15\%$ of EBITDA
- D&A rate: $\delta = 12\%$ of EBITDA  
- Working capital: $\omega = 2\%$ of EBITDA
- Growth rate: $g = 3\%$, WACC: $10\%$

**Step-by-step calculation:**

$$\frac{EV}{EBITDA} = \frac{(1-0.25) + (0.12 \times 0.25) - 0.15 - 0.02}{0.10 - 0.03}$$

$$= \frac{0.75 + 0.03 - 0.15 - 0.02}{0.07} = \frac{0.61}{0.07} = 8.7x$$

**Sensitivity Analysis:**
- If growth increases to 4%: Multiple = $\frac{0.61}{0.06} = 10.2x$
- If CapEx drops to 10%: Multiple = $\frac{0.66}{0.07} = 9.4x$
- If WACC rises to 12%: Multiple = $\frac{0.61}{0.09} = 6.8x$

**Key Takeaway:** Multiples ARE DCF models in disguise!

## Multiples: Market-Based Valuation

**Common Multiples:**
- **EV/EBITDA**: Enterprise Value to EBITDA
- **P/E**: Price to Earnings
- **P/B**: Price to Book Value
- **P/S**: Price to Sales

- The idea is to believe that the multiple of a firm behaves ***similarly*** to the multiple of a comparable firm. Since even across similar firms, multiples can vary, we compare the multiple of a firm to the average (or median) multiple of its peers.
- The best guess of the multiple of your firm is the average (or median) multiple of its peers.
- Who are these peers?

## Triangulation & LLM Applications

**Why Use Both Methods (when available):**

**DCF provides intrinsic value:**
- Forward-looking and company-specific
- Captures unique growth opportunities
- Independent of market sentiment

**Multiples provide market-implied value:**
- Reflects current market conditions
- Quick and comparable across peers
- Incorporates market expectations

# Let's get started

## Accounting Information of TSLA for Business Valuation

In [ ]:
%pip install requests --no-cache-dir

In [ ]:
import requests
from typing import Union, Tuple

def latest_10k_or_10q_text(
    cik: str | int,
    *,
    user_agent: str = "QuickFilingGrabber/2.0 (jfimbett@gmail.com)",
    return_url: bool = False,
) -> Union[str, Tuple[str, str]]:
    """
    Download the newest 10-K or 10-Q for `cik` and return the **raw text**.

    Parameters
    ----------
    cik : str | int
        Company CIK (with or without leading zeros).
    user_agent : str, optional
        REQUIRED by the SEC – include contact info in parentheses.
    return_url : bool, default False
        If True, also return the SEC master-file URL.

    Returns
    -------
    str                    – if return_url is False.
    (str, str) tuple       – if return_url is True (second element is the URL).

    Raises
    ------
    ValueError      – if no recent 10-K/10-Q exists.
    requests.HTTPError – for any HTTP failure.
    """
    cik_raw = str(int(cik))           # strip leading zeros from user input
    cik10   = cik_raw.zfill(10)       # pad back to 10 digits for the API

    # 1) Pull the recent-filings feed -----------------------------------------------
    feed_url = f"https://data.sec.gov/submissions/CIK{cik10}.json"
    hdrs     = {"User-Agent": user_agent, "Accept-Encoding": "gzip, deflate"}
    recent   = requests.get(feed_url, headers=hdrs, timeout=30).json()["filings"]["recent"]

    # 2) Find the first 10-K or 10-Q -------------------------------------------------
    for form, acc in zip(recent["form"], recent["accessionNumber"]):
        if form in {"10-K", "10-Q"}:
            acc_no_dash = acc.replace("-", "")
            txt_url = (
                f"https://www.sec.gov/Archives/edgar/data/{int(cik_raw)}/"
                f"{acc_no_dash}/{acc}.txt"
            )
            raw_txt = requests.get(txt_url, headers=hdrs, timeout=60).text
            break
    else:
        raise ValueError("No recent 10-K or 10-Q found in the feed.")

    return (raw_txt, txt_url) if return_url else raw_txt


# ---------------------------------------------------------------------------
# DEMO: Tesla, Inc. (CIK 0001318605)
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    text, url = latest_10k_or_10q_text("0001318605", return_url=True)
    print(f"Tesla filing URL: {url}")
    print(f"Characters downloaded: {len(text):,}")
    print("\nFirst 500 characters:\n")
    print(text[:500])


In [ ]:
import math
from typing import List

def split_into_chunks(
    text: str,
    n_chunks: int,
    overlap_frac: float = 0.10,
) -> List[str]:
    """
    Split `text` into `n_chunks`, each overlapping the next by `overlap_frac`.

    Parameters
    ----------
    text : str
        The full raw text you want to slice up.
    n_chunks : int
        Desired number of chunks (≥ 2).
    overlap_frac : float, default 0.10
        Fraction of each chunk that should overlap with its successor.
        Must satisfy 0 ≤ overlap_frac < 1.

    Returns
    -------
    List[str]  – `n_chunks` items; the last chunk may be shorter if the
                 math doesn't land exactly on the final character.
    """
    if not (0 <= overlap_frac < 1):
        raise ValueError("overlap_frac must be in the half-open interval [0, 1).")
    if n_chunks < 2:
        return [text]

    L = len(text)
    # Effective step forward each time (non-overlapping part)
    step = L / n_chunks
    overlap = overlap_frac * step
    stride = step - overlap

    # Round to integers for slicing
    step_i   = math.ceil(step)
    stride_i = max(1, math.ceil(stride))

    chunks = []
    start = 0
    for _ in range(n_chunks):
        end = start + step_i
        chunks.append(text[int(start):int(end)])
        start += stride_i
        if start >= L:
            break

    # If rounding left us short of the target count, pad last chunk(s)
    while len(chunks) < n_chunks:
        chunks.append("")
    return chunks


# ---------------------------------------------------------------------------
# EXAMPLE: grab Tesla’s latest 10-K/10-Q, then slice it into 8 chunks
# ---------------------------------------------------------------------------
if __name__ == "__main__":

    chunks = split_into_chunks(text, n_chunks=8, overlap_frac=0.10)
    for i, c in enumerate(chunks, 1):
        print(f"\n--- chunk {i}/{len(chunks)} (len={len(c):,}) ---\n")
        print(c[:500])      # preview first 500 chars of each chunk


In [ ]:
!pip install transformers tqdm 
from transformers import pipeline
from typing import List, Dict, Tuple
from tqdm.auto import tqdm


def extract_financial_values(
    chunks: List[str],
    *,
    # variable → question phrasing
    questions: Dict[str, str] | None = None,
    model_name: str = "deepset/roberta-base-squad2",
    device: int | str | None = None,       # 0, 1, … for GPUs, or "cpu"
    answer_threshold: float = 0.20,        # ignore very low-confidence spans
    progress_desc: str = "Answering questions on chunks",
) -> Dict[str, Dict[str, Tuple]]:
    """
    Ask each `questions[var]` about every chunk and keep the highest-
    confidence answer for every variable.

    Returns
    -------
    {
        "EBIT": {
            "answer": "1,234 million",
            "score": 0.87,
            "chunk_idx": 3
        },
        ...
    }
    """
    if questions is None:
        questions = {
            "EBIT": "What is the company's EBIT?",
            "Taxes": "What is the company's income-tax expense?",
            "Depreciation and Amortization": "What is depreciation and amortization?",
            "Capex": "What were capital expenditures (capex)?",
            "Working Capital": "What is the change in working capital?",
        }

    qa = pipeline("question-answering", model=model_name, device=device)

    # initialise best-answer store
    best: Dict[str, Dict[str, Tuple]] = {
        var: {"answer": None, "score": -1.0, "chunk_idx": None}
        for var in questions
    }

    for idx, chunk in enumerate(tqdm(chunks, desc=progress_desc, unit="chunk")):
        for var, question in questions.items():
            res = qa(question=question, context=chunk)
            if res["score"] >= answer_threshold and res["score"] > best[var]["score"]:
                best[var] = {
                    "answer": res["answer"].strip(),
                    "score": res["score"],
                    "chunk_idx": idx,
                }

    return best


# ─── Example usage ──────────────────────────────────────────────────────────────
chunks = split_into_chunks(text, n_chunks=8, overlap_frac=0.10)
answers = extract_financial_values(chunks, device=0)

for var, info in answers.items():
    print(f"{var:30s}  {info['answer']}  (score={info['score']:.2f}, "
          f"from chunk {info['chunk_idx']})")
